Generate Forecast — Live Production Data
Loads gold data, trains final weekly (LightGBM) and monthly (Prophet, flat growth) models on all available history, and saves a single combined forecast output.

**Input**: gold/erp/battery/phase1_overall_weekly_live.parquet, phase1_overall_monthly_live.parquet
**Output**: gold/forecasts/overall_forecast_latest.json

In [0]:
%run ./_local_config

In [0]:
%pip install lightgbm prophet

In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_gold
import pandas as pd
import datetime
import lightgbm as lgb
from prophet import Prophet

blob_service = get_blob_service(storage_account_name, storage_account_key)

Weekly forecast

In [0]:
gold_weekly = read_gold(blob_service, "live/battery/data/phase1_overall_weekly_live.parquet")
gold_weekly["week_start"] = pd.to_datetime(gold_weekly["week_start"])

feature_cols = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w"]
target_col = "total_units_sold"

model_data = gold_weekly.dropna(subset=feature_cols + [target_col]).copy()

final_weekly_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6)
final_weekly_model.fit(model_data[feature_cols], model_data[target_col])

last_week_start = gold_weekly["week_start"].max()
next_week_start = last_week_start + pd.Timedelta(weeks=1)
next_week_end = next_week_start + pd.Timedelta(days=6)

lag_4w_value = gold_weekly[gold_weekly["week_start"] == next_week_start - pd.Timedelta(weeks=4)]["total_units_sold"].values
lag_4w_value = lag_4w_value[0] if len(lag_4w_value) > 0 else None

next_week_features = pd.DataFrame([{
    "week_of_year": next_week_start.isocalendar()[1],
    "month": next_week_start.month,
    "contains_month_end": int(next_week_start.month != next_week_end.month),
    "lag_4w": lag_4w_value,
    "rolling_avg_4w": gold_weekly["total_units_sold"].tail(4).mean(),
}])

weekly_prediction = final_weekly_model.predict(next_week_features[feature_cols])[0]
print(f"Weekly forecast — week of {next_week_start.date()}: {weekly_prediction:.0f}")

Monthly forecast

In [0]:
gold_monthly = read_gold(blob_service, "live/battery/data/phase1_overall_monthly_live.parquet")
gold_monthly["month_start"] = pd.to_datetime(gold_monthly["month_start"])

monthly_prophet_df = gold_monthly[["month_start", "total_units_sold"]].rename(
    columns={"month_start": "ds", "total_units_sold": "y"}
)

m_final = Prophet(yearly_seasonality=True, weekly_seasonality=False, growth='flat')
m_final.fit(monthly_prophet_df)

future_final = m_final.make_future_dataframe(periods=3, freq="MS")
forecast_final = m_final.predict(future_final)
forecast_final[["yhat", "yhat_lower", "yhat_upper"]] = forecast_final[["yhat", "yhat_lower", "yhat_upper"]].clip(lower=0)

monthly_forecast = forecast_final[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3)
print(monthly_forecast)

Combine and save

In [0]:
# import json

# forecast_output = {
#     "generated_at": datetime.datetime.now().isoformat(),
#     "weekly": {
#         "week_start": next_week_start.strftime("%Y-%m-%d"),
#         "predicted_units": round(float(weekly_prediction))
#     },
#     "monthly": [
#         {
#             "month_start": row["ds"].strftime("%Y-%m-%d"),
#             "predicted_units": round(float(row["yhat"])),
#             "lower_bound": round(float(row["yhat_lower"])),
#             "upper_bound": round(float(row["yhat_upper"]))
#         }
#         for _, row in monthly_forecast.iterrows()
#     ]
# }

# print(json.dumps(forecast_output, indent=2))

# blob_client = blob_service.get_blob_client(container="gold", blob="live/forecasts/battery/overall_forecast_latest.json")
# blob_client.upload_blob(json.dumps(forecast_output, indent=2), overwrite=True)
# print("Saved to gold/live/forecasts/battery/overall_forecast_latest.json")

In [0]:
import json
import datetime

from src.io.storage import append_json_history, read_json_history

today_str = datetime.date.today().isoformat()
today = pd.Timestamp(datetime.date.today())

# ── Weekly: append to history ──────────────────────────────
weekly_record = {
    "generated_date": today_str,
    "week_start": next_week_start.strftime("%Y-%m-%d"),
    "week_end": (next_week_start + pd.Timedelta(days=6)).strftime("%Y-%m-%d"),
    "predicted_units": round(float(weekly_prediction))
}

weekly_history = append_json_history(blob_service, [weekly_record], "live/battery/forecasts/history/weekly_forecast_history.json")

# ── Monthly: append to history ─────────────────────────────
monthly_records = [
    {
        "generated_date": today_str,
        "month_start": row["ds"].strftime("%Y-%m-%d"),
        "predicted_units": round(float(row["yhat"])),
        "lower_bound": round(float(row["yhat_lower"])),
        "upper_bound": round(float(row["yhat_upper"]))
    }
    for _, row in monthly_forecast.iterrows()
]

monthly_history = append_json_history(blob_service, monthly_records, "live/battery/forecasts/history/monthly_forecast_history.json")

print(f"Weekly history: {len(weekly_history)} total records")
print(f"Monthly history: {len(monthly_history)} total records")

# ── Weekly: compute and save active (not-yet-ended) forecasts ──
weekly_df = pd.DataFrame(weekly_history)
weekly_df["week_end"] = pd.to_datetime(weekly_df["week_end"])
weekly_df["generated_date"] = pd.to_datetime(weekly_df["generated_date"])

active_weekly = weekly_df[weekly_df["week_end"] >= today]
active_weekly = active_weekly.sort_values("generated_date").drop_duplicates(subset="week_start", keep="last")
active_weekly = active_weekly.sort_values("week_start")

active_weekly_records = active_weekly.to_dict(orient="records")
for r in active_weekly_records:
    r["week_end"] = r["week_end"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/weekly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_weekly_records, indent=2), overwrite=True)
print(f"Active weekly forecasts: {len(active_weekly_records)}")
print(active_weekly_records)

# ── Monthly: compute and save active (not-yet-ended) forecasts ──
monthly_df = pd.DataFrame(monthly_history)
monthly_df["month_start"] = pd.to_datetime(monthly_df["month_start"])
monthly_df["generated_date"] = pd.to_datetime(monthly_df["generated_date"])
monthly_df["month_end"] = monthly_df["month_start"] + pd.offsets.MonthEnd(0)

active_monthly = monthly_df[monthly_df["month_end"] >= today]
active_monthly = active_monthly.sort_values("generated_date").drop_duplicates(subset="month_start", keep="last")
active_monthly = active_monthly.sort_values("month_start")

active_monthly_records = active_monthly.drop(columns=["month_end"]).to_dict(orient="records")
for r in active_monthly_records:
    r["month_start"] = r["month_start"].strftime("%Y-%m-%d")
    r["generated_date"] = r["generated_date"].strftime("%Y-%m-%d")

blob_client = blob_service.get_blob_client(container="gold", blob="live/battery/forecasts/active/monthly_forecast_active.json")
blob_client.upload_blob(json.dumps(active_monthly_records, indent=2), overwrite=True)
print(f"Active monthly forecasts: {len(active_monthly_records)}")
print(active_monthly_records)

In [0]:
import pandas as pd
import io

def save_history_as_excel(blob_service, history_records, blob_path):
    df = pd.DataFrame(history_records)
    buffer = io.BytesIO()
    df.to_excel(buffer, index=False, engine="openpyxl")
    buffer.seek(0)
    blob_client = blob_service.get_blob_client(container="gold", blob=blob_path)
    blob_client.upload_blob(buffer, overwrite=True)
    print(f"Saved Excel: {blob_path} ({len(df)} rows)")

save_history_as_excel(blob_service, weekly_history, "live/battery/forecasts/history/weekly_forecast_history.xlsx")
save_history_as_excel(blob_service, monthly_history, "live/battery/forecasts/history/monthly_forecast_history.xlsx")

In [0]:
from src.io.storage import get_blob_service, read_gold, append_json_history, save_history_as_excel

In [0]:
count = save_history_as_excel(blob_service, weekly_history, "live/battery/forecasts/history/weekly_forecast_history.xlsx")
print(f"Saved weekly Excel: {count} rows")

count = save_history_as_excel(blob_service, monthly_history, "live/battery/forecasts/history/monthly_forecast_history.xlsx")
print(f"Saved monthly Excel: {count} rows")